# Results for model: qwen/qwen2.5-7b-instruct

In [ ]:
import polars as pl
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

# Load data
df = pl.read_parquet('./jane_street_data/train.parquet')

# Feature Engineering
df = df.with_column(
    pl.col('feature_00').rolling_quantile(window_size=100, quantile=0.85, closed='both').alias('top_quantile_feature_00')
)

# Split into features and target
X = df.select(['feature_00', 'top_quantile_feature_00'])
y = df['responder_6']

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Train XGBoost Regressor
model = XGBRegressor(n_estimators=1000, learning_rate=0.05, n_jobs=-1)
model.fit(X_train.to_pandas(), y_train.to_pandas(), eval_set=(X_val.to_pandas(), y_val.to_pandas()), early_stopping_rounds=50, verbose=False)

# Predict and evaluate
y_pred = model.predict(X_val.to_pandas())
mse = mean_squared_error(y_val.to_pandas(), y_pred)
print(f'Mean Squared Error: {mse}')

# Save model (optional)
model.save_model('xgboost_model.json')